<a href="https://colab.research.google.com/github/SindhuBangaru/ai-mentor-portfolio/blob/main/Day6_PlaceProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install -q google-genai pydantic
import os, getpass

os.environ['GEMINI_API_KEY'] = getpass.getpass(
    'Enter NEW Gemini API key: '
)

Enter NEW Gemini API key: ··········


In [35]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [36]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [37]:
with open('../content/sample_data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Ravi Kumar — 6 skills
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


In [38]:
# Empty string
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890","education":[{"degree":"Master of Science in Computer Science","institution":"University of Example","year":2022},{"degree":"Bachelor of Engineering in Software Engineering","institution":"Tech Institute","year":2020}],"skills":["Python","Java","C++","JavaScript","React","SQL","AWS","Docker","Machine Learning"],"projects":["E-commerce Platform Development","AI Chatbot Integration","Cloud Migration Project"],"experience_years":3.5}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.


In [39]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [40]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

# Test on one URL
test_url = 'https://amazon.jobs/en/jobs/10429151/salesforce-developer-dsp-tech-last-mile'
text = fetch_jd(test_url)
if text:
    print(f'Got {len(text)} chars')
    print(text[:300])
else:
    print('Scrape blocked. Will use cached set.')

  Scrape failed for https://amazon.jobs/en/jobs/10429151/salesforce-developer-dsp-tech-last-mile: ('Received response with content-encoding: zstd, but failed to decode it.', ZstdError('cannot use a decompressobj multiple times'))
Scrape blocked. Will use cached set.


In [41]:
def normalise_jd(text: str) -> JD:
    """Send JD text to Gemini, get structured JD JSON back."""
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract a JD JSON from this text:\n\n{text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )
    return JD.model_validate_json(resp.text)

# Test on one JD text
if text:
    jd = normalise_jd(text)
    print(jd.model_dump_json(indent=2))

In [45]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://amazon.jobs/en/jobs/10429151/salesforce-developer-dsp-tech-last-mile',
    'https://amazon.jobs/en/jobs/10393375/associate-systems-engineer-windows-region-services',
    'https://www.accenture.com/in-en/careers/jobdetails?id=ATCI-5556164-S2025420_en&title=Custom+Software+Engineer',
    'https://careers.wipro.com/job/Practice-Specialist-Industry-Cloud-&amp;-Digital/153629-en_US',
    'https://www.google.com/about/careers/applications/jobs/results/87610860276458182-software-engineering-manager-ii-google-cloudO_url_5',
]

CACHE = pathlib.Path('../data/jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✓ Amazon — Salesforce Developer
  ✓ Amazon — Associate Systems Engineer - Windows , Region Services
  ✓ Accenture — Custom Software Engineer
  ✓ Wipro Limited — Practice Specialist - Industry Cloud & Digital
  ✓ Google — Software Engineering Manager II, Google Cloud

Processed 5 JDs

Amazon - Salesforce Developer
  Must: ['Experience in automating, deploying, and supporting infrastructure', 'Experience programming with at least one modern language such as Python, Ruby, Golang, Java, C++, C#, Rust', 'Experience with Linux/Unix', 'Salesforce.com', 'Force.com', 'VisualForce', 'APEX', 'Lightning Web Components', 'Web Services', 'APIs', 'Salesforce configuration (user management, security, custom objects, page layouts, validations, workflows, flows, process builders)', 'System integrations (Salesforce ecosystem, backend services, AWS technologies, third-party applications)', 'Planning and analyzing requirements for system enhancements']
  Nice: ['Experience with CI/CD pipelines build proc

In [49]:
OUT = pathlib.Path('data/jds.jsonl')
OUT.parent.mkdir(exist_ok=True)
with open(OUT, 'w') as f:
    for jd in jds:
        f.write(jd.model_dump_json() + '\n')
print(f'Wrote {len(jds)} JDs to {OUT}')

# Verify the file
with open(OUT) as f:
    for line in f:
        d = json.loads(line)
        print(f'  {d["company"]:20} | {d["role"]:30} | {len(d["must_have_skills"])} must-haves')

Wrote 5 JDs to data/jds.jsonl
  Amazon               | Salesforce Developer           | 13 must-haves
  Amazon               | Associate Systems Engineer - Windows , Region Services | 2 must-haves
  Accenture            | Custom Software Engineer       | 1 must-haves
  Wipro Limited        | Practice Specialist - Industry Cloud & Digital | 15 must-haves
  Google               | Software Engineering Manager II, Google Cloud | 7 must-haves
